In [ ]:
# Import required libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print('Python', sys.version)
print('numpy', np.__version__, 'pandas', pd.__version__)

In [ ]:
# Ensure ucimlrepo is installed (will be a no-op if already present)
import subprocess, sys
try:
    import ucimlrepo
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ucimlrepo'])
    import ucimlrepo
print('ucimlrepo version:', getattr(ucimlrepo, '__version__', 'unknown'))

In [ ]:
# Fetch dataset
from ucimlrepo import fetch_ucirepo
air_quality = fetch_ucirepo(id=360)
print(type(air_quality))
print('Has attributes:', [a for a in dir(air_quality) if not a.startswith('_')])
# show summary of data structure
data_preview = getattr(air_quality, 'data', air_quality)
print('data type:', type(data_preview))
try:
    import numpy as _np
    _arr = _np.asarray(data_preview)
    print('data ndarray shape:', getattr(_arr, 'shape', None))
except Exception as e:
    print('could not convert data to ndarray:', e)

{'data': {'ids': None,
  'features':            Date      Time  CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  \
  0     3/10/2004  18:00:00     2.6         1360       150      11.9   
  1     3/10/2004  19:00:00     2.0         1292       112       9.4   
  2     3/10/2004  20:00:00     2.2         1402        88       9.0   
  3     3/10/2004  21:00:00     2.2         1376        80       9.2   
  4     3/10/2004  22:00:00     1.6         1272        51       6.5   
  ...         ...       ...     ...          ...       ...       ...   
  9352   4/4/2005  10:00:00     3.1         1314      -200      13.5   
  9353   4/4/2005  11:00:00     2.4         1163      -200      11.4   
  9354   4/4/2005  12:00:00     2.4         1142      -200      12.4   
  9355   4/4/2005  13:00:00     2.1         1003      -200       9.5   
  9356   4/4/2005  14:00:00     2.2         1071      -200      11.9   
  
        PT08.S2(NMHC)  NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  \
  0              1046      

In [ ]:
# Robust conversion of the fetched object to a pandas DataFrame
data_obj = getattr(air_quality, 'data', air_quality)
feature_names = getattr(air_quality, 'feature_names', getattr(air_quality, 'feature_names_', None))
# If the returned object is already a DataFrame, copy it
if isinstance(data_obj, pd.DataFrame):
    df = data_obj.copy()
else:
    # If the object is a dict-like, let pandas handle it
    if isinstance(data_obj, dict):
        df = pd.DataFrame(data_obj)
    else:
        arr = np.asarray(data_obj)
        # structured array with named fields -> pandas will use field names
        if getattr(arr.dtype, 'names', None):
            df = pd.DataFrame(arr)
        elif arr.ndim == 1:
            col = feature_names[0] if feature_names and len(feature_names) > 0 else 'col0'
            df = pd.DataFrame(arr, columns=[col])
        else:
            ncols = arr.shape[1]
            if feature_names and len(feature_names) == ncols:
                cols = list(feature_names)
            else:
                cols = [ (feature_names[i] if feature_names and i < len(feature_names) else f'col{i}') for i in range(ncols) ]
            df = pd.DataFrame(arr, columns=cols)
# Inspect the dataframe
print('DataFrame shape:', df.shape)
print('First 10 columns (if many):', df.columns.tolist()[:10])
display(df.head())

In [ ]:
# Safe column selection: try exact names, then substring matches (case-insensitive)
wanted = ['CO(GT)','C6H6(GT)','NOx(GT)','NO2(GT)']
found = []
cols = df.columns.tolist()
for w in wanted:
    if w in cols:
        found.append(w)
    else:
        # try case-insensitive substring match
        matches = [c for c in cols if w.lower().replace('(','').replace(')','') in c.lower().replace('(', '').replace(')', '') or w.split('(')[0].lower() in c.lower()]
        if matches:
            found.append(matches[0])
        else:
            print('Column not found for', w)
# Use the found columns; if fewer than 2, warn and stop
if len(found) < 2:
    raise ValueError('Not enough matching columns found for anomaly detection. Found: ' + str(found))
df_sel = df[found].copy()
# convert to numeric and impute missing values with column medians
for c in df_sel.columns:
    df_sel[c] = pd.to_numeric(df_sel[c], errors='coerce')
df_sel = df_sel.fillna(df_sel.median())
print('Prepared data shape for model:', df_sel.shape)
display(df_sel.head())

In [ ]:
# Model hyperparameters
n_estimators = 100
contamination = 0.1
sample_size = 256
print('n_estimators, contamination, sample_size:', n_estimators, contamination, sample_size)

In [ ]:
# Train IsolationForest (use max_samples parameter, and ensure it's <= n_samples)
from sklearn.ensemble import IsolationForest
n_samples = len(df_sel)
max_samples = min(sample_size, n_samples)
model = IsolationForest(n_estimators=n_estimators, contamination=contamination, max_samples=max_samples, random_state=42)
model.fit(df_sel)
print('Model fitted on', n_samples, 'rows; max_samples used =', max_samples)

In [ ]:
# Predict and attach anomaly labels to the original dataframe
labels = model.predict(df_sel)
# IsolationForest outputs -1 for outliers, 1 for inliers. Map to boolean anomaly column
anomaly_bool = (labels == -1)
# Attach to original df (align by index)
df = df.copy()
df['anomaly'] = False
df.loc[df_sel.index, 'anomaly'] = anomaly_bool
print('Anomalies found:', int(df['anomaly'].sum()))
display(df.head())

In [ ]:
#plot1; CO vs C6H6
sns.scatterplot(data=df, x=found[0], y=found[1], hue='anomaly', palette={False: 'blue', True: 'red'}, alpha=0.6)
plt.title('Anomaly Detection: CO vs C6H6')  
